In [57]:
import pathlib

import pandas as pd
from torch.utils.data import DataLoader
import altair as alt

import trapallada.data

So that imported modules are automatically reloaded

In [58]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Images

The *base* file name for the data

In [59]:
# base_filename = pathlib.Path('small.pt')
base_filename = pathlib.Path.home() / 'Sync' / 'research' / 'light curves' / 'small.pt'

`Dataset` is loaded: 18,000 gray-scale (i.e., single-channel) images

In [60]:
ds = trapallada.data.ImageDataset(
    base_filename.with_stem(base_filename.stem + '_imgs'),
    base_filename.with_stem(base_filename.stem + '_labels').with_suffix('.parquet'),
)
ds.imgs.shape

torch.Size([18000, 1, 30, 250])

There is an internal `pd.DataFrame` storing all the metadata

In [61]:
ds.df

,label,std,file name
0,Constant,1.0,Constant/cLC_s1_793.jpg
1,Constant,3.0,Constant/cLC_s3_524.jpg
2,Constant,3.0,Constant/cLC_s3_956.jpg
3,Constant,3.0,Constant/cLC_s3_283.jpg
4,Constant,5.0,Constant/cLC_s5_666.jpg
...,...,...,...
17995,AM,3.0,AM/AM_s3_613.jpg
17996,AM,1.0,AM/AM_s1_303.jpg
17997,AM,2.0,AM/AM_s2_894.jpg
17998,AM,10.0,AM/AM_s10_750.jpg


Using a `DataLoader`

In [62]:
dl = DataLoader(ds, batch_size=5)
dl

How many things are output by every batch?

In [63]:
batch = next(iter(dl))
len(batch)

3

- Indexes

- Images

- Labels

In [64]:
batch[0], batch[1].shape, batch[2]

(tensor([0, 1, 2, 3, 4]),
 torch.Size([5, 1, 30, 250]),
 tensor([1, 1, 1, 1, 1], dtype=torch.int8))

# Signals

Input directory

In [65]:

input_directory = pathlib.Path.cwd().parent / 'data'
assert input_directory.exists()

Individual files with

- the actual signals, and

- a `DataFrame` with metadata.

In [66]:
base_filename = '4_classes_modulated'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

A *PyTorch* `Dataset`

In [67]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
len(ds)

12983

It encompasses a *PyTorch* `Tensor` for the signals

In [68]:
ds.signals.shape

torch.Size([12983, 1, 10000])

and a *Pandas* `DataFrame`

In [69]:
ds.metadata_df.sample(n=10)

,label,SNR,color,file name
12034,noise,NaN,0.10792,noise/noise_color0.10792.mat
5887,AM,31.620,None,AM/AM_SNR31.62_Simul341.mat
11967,carrier,31.620,None,singlePeriod/carrier_SNR31.62_Simul376.mat
12490,noise,NaN,1.5397,noise/noise_color1.5397.mat
12261,noise,NaN,0.72206,noise/noise_color0.72206.mat
10024,carrier,10.000,None,singlePeriod/carrier_SNR10_Simul852.mat
3374,FM,31.620,None,FM/FM_SNR31.62_Simul264.mat
192,FM,3.162,None,FM/FM_SNR3.162_Simul869.mat
7932,AM,10.000,None,AM/AM_SNR10_Simul709.mat
12307,noise,NaN,1.533,noise/noise_color1.533.mat


In [71]:
mean = ds.signals.mean(axis=-1)
std = ds.signals.std(axis=-1)
mean, std

(tensor([[-1.9975e-01],
         [-1.0158e-01],
         [ 1.4231e-04],
         ...,
         [ 2.2189e-02],
         [ 3.9760e-01],
         [ 2.2086e-02]], dtype=torch.float64),
 tensor([[0.8860],
         [0.7669],
         [0.7920],
         ...,
         [0.9930],
         [2.7855],
         [1.1968]], dtype=torch.float64))

One can also *normalize* the `DataSet`

In [73]:
normalized_ds = trapallada.data.SignalsDataset(signals_file, metadata_file,normalize=True)
mean = normalized_ds.signals.mean(axis=-1)
std = normalized_ds.signals.std(axis=-1)
mean, std

(tensor([[ 1.0658e-17],
         [ 4.9738e-18],
         [-4.9738e-18],
         ...,
         [ 5.5067e-18],
         [ 3.7303e-17],
         [ 1.3500e-17]], dtype=torch.float64),
 tensor([[1.0000],
         [1.0000],
         [1.0000],
         ...,
         [1.0000],
         [1.0000],
         [1.0000]], dtype=torch.float64))

Using a `DataLoader`

In [ ]:
dl = DataLoader(ds, batch_size=5)
dl

How many things are output by every batch?

In [ ]:
batch = next(iter(dl))
len(batch)

- Indexes

- Images

- Labels

In [ ]:
batch[0], batch[1].shape, batch[2]

Let us plot the first

In [ ]:
n_rows = 1_000

rows of the first signal in the batch

In [ ]:
df = pd.DataFrame(batch[1][0].squeeze(), columns=['signal'])
alt.Chart(df.iloc[:n_rows].reset_index()).mark_line().encode(
    x='index', y='signal'
)


Some random signals

In [ ]:
sample_metadata_df = ds.metadata_df.sample(n=75, random_state=42)
sample_metadata_df

A `DataFrame` is built for them

In [ ]:
sample_signals = ds.signals[sample_metadata_df.index.tolist()].squeeze().T
df = pd.DataFrame(sample_signals, columns=sample_metadata_df['file name'].str.extract(r'.*/(.*).(?:jpg|mat)')[0])
df

and plotted

In [ ]:
df.columns = df.columns.str.replace('.', '')
alt.Chart(
    df.iloc[:n_rows].reset_index()
).transform_fold(
    df.columns.to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)